# Eksperimen penutup — kredit non-keluarga & usaha RT (JES)
Notebook ini **tidak butuh file .dta IFLS**. Inputnya hanya dua CSV yang sudah ada:
`umkm_ifls_v2.csv` (output run FULL) dan `umkm_ifls3_placebo.csv`.

| Eksperimen | Pertanyaan yang dijawab | Output |
|---|---|---|
| X1 Repeated cross-fitting (S split) | Apakah selisih antar-desain fold / trimming / non-mover hanya noise satu split? | `X1_repeated.csv`, `X1_splits_raw.csv` |
| X2 Overlap | Apakah efek bergantung pada observasi propensity ekstrem? (trimming grid + overlap-weighted AIPW) | `X2_overlap.csv` |
| X3 Placebo pre-trend | Apakah kredit 2014 "memprediksi" pertumbuhan usaha 2000→2007? (negative-control outcome yang tidak memengaruhi permintaan kredit) | `X3_pretrend.csv` |
| X4 Subgrup | Apakah efek terkonsentrasi pada usaha muda / yang belum ada di 2000? (menjelaskan hasil history-augmented) | `X4_subgroups.csv` |
| X5 Sensitivitas Cinelli–Hazlett | Seberapa kuat konfounder (relatif thd pinjaman 2007) untuk menghapus efek? | `X5_sensitivity.csv` |
| X6 Heterogenitas berulang | Apakah BLP bertanda negatif stabil lintas split? (median p, Chernozhukov et al. 2025) | `X6_blp_repeated.csv` |

Cara pakai: Add Input → dataset/output yang berisi kedua CSV. CPU. `MODE="FULL"` → Save & Run All (±1–2 jam).

In [ ]:
import os
for _v in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ[_v] = "1"
import sys, time, json, math, glob, warnings
import numpy as np, pandas as pd
from scipy import stats
from joblib import Parallel, delayed
from sklearn.model_selection import KFold
from sklearn.cluster import KMeans
from sklearn.ensemble import (HistGradientBoostingClassifier, HistGradientBoostingRegressor,
                              RandomForestClassifier, RandomForestRegressor)
warnings.filterwarnings("ignore")
try:
    import lightgbm as lgb; HAS_LGB = True
except Exception:
    HAS_LGB = False

MODE    = "FULL"                       # "QUICK" untuk uji cepat
S_SPLIT = 20 if MODE == "FULL" else 2  # jumlah split cross-fitting berulang
N_FOLDS = 5
SEED    = 2026                         # sama dengan run utama -> split ke-0 mereproduksi Tabel 3
PS_CLIP = 0.01
N_JOBS  = -1
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("./x_out")
os.makedirs(OUT, exist_ok=True)
T0 = time.time()
def log(*a): print(f"[{(time.time()-T0)/60:6.1f} mnt]", *a, flush=True)
log("MODE", MODE, "| S_SPLIT", S_SPLIT, "| lightgbm", HAS_LGB, "| CPU", os.cpu_count())

## Fungsi inti (disalin persis dari notebook utama agar split ke-0 identik dengan Tabel 3)

In [ ]:
def make_clf(kind="gbm", fast=False):
    it = 100 if fast else 300
    if kind == "rf":
        return RandomForestClassifier(n_estimators=200 if not fast else 100, min_samples_leaf=10, n_jobs=1, random_state=0)
    if HAS_LGB:
        return lgb.LGBMClassifier(n_estimators=it, learning_rate=0.1 if fast else 0.05, num_leaves=15,
                                  min_child_samples=20, subsample=0.8, subsample_freq=1,
                                  colsample_bytree=0.8, reg_lambda=1.0, verbose=-1, n_jobs=1)
    return HistGradientBoostingClassifier(max_iter=it, learning_rate=0.1 if fast else 0.05,
                                          max_leaf_nodes=15, min_samples_leaf=20, l2_regularization=1.0,
                                          early_stopping=False)

def make_reg(kind="gbm", fast=False):
    it = 100 if fast else 300
    if kind == "rf":
        return RandomForestRegressor(n_estimators=200 if not fast else 100, min_samples_leaf=10, n_jobs=1, random_state=0)
    if HAS_LGB:
        return lgb.LGBMRegressor(n_estimators=it, learning_rate=0.1 if fast else 0.05, num_leaves=15,
                                 min_child_samples=20, subsample=0.8, subsample_freq=1,
                                 colsample_bytree=0.8, reg_lambda=1.0, verbose=-1, n_jobs=1)
    return HistGradientBoostingRegressor(max_iter=it, learning_rate=0.1 if fast else 0.05,
                                         max_leaf_nodes=15, min_samples_leaf=20, l2_regularization=1.0,
                                         early_stopping=False)

def make_folds(scheme, groups, region_xy, k=5, seed=0, blocks_per_fold=2):
    # groups: label wilayah per observasi; region_xy: DataFrame index=label, kolom [x,y]
    n = len(groups); rng = np.random.default_rng(seed)
    if scheme == "random":
        return list(KFold(k, shuffle=True, random_state=seed).split(np.arange(n)))
    ug = np.asarray(region_xy.index)
    if scheme == "group":
        perm = rng.permutation(len(ug)); fold_of = dict(zip(ug[perm], np.arange(len(ug)) % k))
    elif scheme == "spatial":
        nb = min(len(ug), k * blocks_per_fold)
        lab = KMeans(nb, n_init=5, random_state=seed).fit_predict(region_xy[["x", "y"]].values)
        bperm = rng.permutation(nb); fold_of_block = {b: i % k for i, b in enumerate(bperm)}
        fold_of = {g: fold_of_block[l] for g, l in zip(ug, lab)}
    else:
        raise ValueError(scheme)
    f = np.array([fold_of[g] for g in groups])
    return [(np.where(f != j)[0], np.where(f == j)[0]) for j in range(k) if (f == j).any()]

def cluster_se(psi_centered, groups, w=None):
    # SE rata-rata (berbobot) dari skor terpusat, klaster pada groups
    w = np.ones_like(psi_centered) if w is None else w
    s = pd.Series(w * psi_centered).groupby(np.asarray(groups)).sum().values
    G = len(s)
    return math.sqrt(G / (G - 1) * np.sum(s ** 2)) / np.sum(w)

def _fit_pred(model, Xtr, ytr, Xte, binary):
    if binary:
        if ytr.min() == ytr.max():
            return np.full(len(Xte), float(ytr.mean()))
        return model.fit(Xtr, ytr).predict_proba(Xte)[:, 1]
    return model.fit(Xtr, ytr).predict(Xte)

def dml_aipw(Y, D, X, folds, binary_y, groups, w=None, clip=PS_CLIP, kind="gbm", fast=False):
    n = len(Y); e = np.zeros(n); m1 = np.zeros(n); m0 = np.zeros(n)
    for tr, te in folds:
        e[te] = _fit_pred(make_clf(kind, fast), X[tr], D[tr], X[te], True)
        for d, arr in ((1, m1), (0, m0)):
            sel = tr[D[tr] == d]
            mdl = make_clf(kind, fast) if binary_y else make_reg(kind, fast)
            arr[te] = _fit_pred(mdl, X[sel], Y[sel], X[te], binary_y)
    extreme = float(np.mean((e < clip) | (e > 1 - clip)))
    ec = np.clip(e, clip, 1 - clip)
    psi = m1 - m0 + D * (Y - m1) / ec - (1 - D) * (Y - m0) / (1 - ec)
    w_ = np.ones(n) if w is None else w
    ate = np.sum(w_ * psi) / np.sum(w_)
    se_iid = math.sqrt(np.sum(w_ ** 2 * (psi - ate) ** 2)) / np.sum(w_)
    se_cl = cluster_se(psi - ate, groups, w_)
    # ATT (AIPW)
    p = np.sum(w_ * D) / np.sum(w_)
    num = D * (Y - m0) - (1 - D) * ec / (1 - ec) * (Y - m0)
    att = np.sum(w_ * num) / np.sum(w_ * D)
    att_se = cluster_se((num - D * att) / p, groups, w_)
    return dict(ate=ate, se_iid=se_iid, se_cl=se_cl, att=att, att_se=att_se,
                extreme=extreme, psi=psi, e=ec, m1=m1, m0=m0)

def dr_learner(psi, X, folds, kind="gbm", fast=False):
    tau = np.zeros(len(psi))
    lo, hi = np.quantile(psi, [0.005, 0.995]); y = np.clip(psi, lo, hi)  # winsor ringan utk stabilitas
    for tr, te in folds:
        tau[te] = make_reg(kind, fast).fit(X[tr], y[tr]).predict(X[te])
    return tau

def ols_cluster(y, Xmat, groups, w=None):
    # WLS + SE klaster (CR1). Xmat sudah memuat konstanta.
    w = np.ones(len(y)) if w is None else w
    sw = np.sqrt(w); Xw = Xmat * sw[:, None]; yw = y * sw
    XtX_inv = np.linalg.pinv(Xw.T @ Xw)
    b = XtX_inv @ Xw.T @ yw
    u = yw - Xw @ b
    meat = np.zeros((Xmat.shape[1],) * 2)
    for g in np.unique(groups):
        idx = groups == g; s = Xw[idx].T @ u[idx]; meat += np.outer(s, s)
    n, k = Xmat.shape; G = len(np.unique(groups))
    V = XtX_inv @ meat @ XtX_inv * (G / (G - 1)) * ((n - 1) / (n - k))
    return b, np.sqrt(np.clip(np.diag(V), 0, None)), G

def blp_test(psi, tau_hat, groups, w=None):
    # psi_i = a + b*(tau_hat_i - mean) ; b>0 & signifikan => heterogenitas terdeteksi
    t = tau_hat - np.average(tau_hat, weights=w)
    Xm = np.column_stack([np.ones_like(t), t])
    b, se, G = ols_cluster(psi, Xm, np.asarray(groups), w)
    pval = 2 * (1 - stats.t.cdf(abs(b[1] / se[1]), df=G - 1)) if se[1] > 0 else np.nan
    return dict(ate_blp=b[0], beta_het=b[1], se_het=se[1], p_het=pval)

def gates(psi, tau_hat, groups, q=5, w=None):
    cut = pd.qcut(tau_hat, q, labels=False, duplicates="drop")
    rows = []
    for g in np.unique(cut):
        i = cut == g; ww = None if w is None else w[i]
        m = np.average(psi[i], weights=ww)
        rows.append(dict(group=int(g) + 1, gate=m, se=cluster_se(psi[i] - m, np.asarray(groups)[i], ww), n=int(i.sum())))
    return pd.DataFrame(rows)

def mundlak(X, g):
    df = pd.DataFrame(X); s = df.groupby(g).transform("sum"); c = pd.Series(g).map(pd.Series(g).value_counts()).values
    loo = (s.values - X) / np.maximum(c - 1, 1)[:, None]
    return np.column_stack([loo, np.log(c)])


def find_csv(name):
    roots = ["/kaggle/input", os.getcwd()]
    hits = []
    for r in roots:
        hits += glob.glob(os.path.join(r, "**", name), recursive=True)
    hits = [h for h in hits if "/x_out/" not in h]
    if not hits: raise FileNotFoundError(f"{name} tidak ditemukan di Input")
    return sorted(hits, key=len)[0]

def key(s): return s.astype(str).str.strip().str.replace(r"\.0$", "", regex=True).str.lstrip("0")

## Data

In [ ]:
dat = pd.read_csv(find_csv("umkm_ifls_v2.csv"))
pl = pd.read_csv(find_csv("umkm_ifls3_placebo.csv"), dtype={"hhid14": str})
COVS = ["ihs_laba07", "pekerja07", "ihs_aset07", "umur_usaha07", "n_usaha07", "usaha_di_rumah07",
        "pinjam07", "ihs_pinjam07", "log_pce07", "hhsize07", "kepala_perempuan", "usia_kepala",
        "pendidikan_kepala", "perkotaan07", "n_lembaga_kredit_desa07", "jarak_lembaga_min07"]
COVS = [c for c in COVS if c in dat] + [c for c in dat.columns if c.endswith("_miss")]
OUTS = {"usaha_tutup": True, "ihs_laba": False, "tenaga_kerja": False}
D = dat["kredit"].values.astype(int); W = dat["weight"].values.astype(float)
G = dat["kab_kode"].values; P = dat["prov_kode"].values

def feat(df, spec="C", extra=None):
    X = df[COVS].values.astype(float)
    if extra is not None: X = np.column_stack([X, extra])
    if spec == "A": return X
    xy = df[["lon", "lat"]].values.astype(float)
    if spec == "B": return np.column_stack([X, xy])
    return np.column_stack([X, xy, mundlak(df[COVS].values.astype(float), df["kab_kode"].values)])

def rxy(df):
    return df.groupby("kab_kode")[["lon", "lat"]].mean().rename(columns={"lon": "x", "lat": "y"})

def fit_one(df, y, binary, scheme, seed, w=None, X=None, kind="gbm"):
    X = feat(df) if X is None else X
    f = make_folds(scheme, df["kab_kode"].values, rxy(df), N_FOLDS, seed=seed)
    return dml_aipw(y, df["kredit"].values.astype(int), X, f, binary, df["kab_kode"].values,
                    df["weight"].values.astype(float) if w is None else w, kind=kind)

def median_agg(est, se):
    est, se = np.asarray(est), np.asarray(se); m = np.median(est)
    return m, float(np.median(np.sqrt(se ** 2 + (est - m) ** 2)))

def cl_mean(x, g, w):
    m = np.average(x, weights=w); return m, cluster_se(x - m, g, w)

XC = feat(dat)
log(f"N={len(dat)} | kab/kota={len(np.unique(G))} | prov={len(np.unique(P))} | kovariat={len(COVS)}")

## X1. Cross-fitting berulang: desain fold utama & robustness kunci
Setiap split menyimpan ATE, SE, ATT, propensity, dan skor ψ (untuk X2, X4, X6).

In [ ]:
CHECKS = ["main", "trim10", "nonmover", "unweighted"]
def one_split(s):
    seed = SEED + s * 7919 if s > 0 else SEED
    rows, keep = [], {}
    for scheme in ["random", "group", "spatial"]:
        for o, b in OUTS.items():
            y = dat[o].values.astype(float)
            r = fit_one(dat, y, b, scheme, seed, X=XC)
            rows.append(dict(split=s, scheme=scheme, outcome=o, check="main", ate=r["ate"], se=r["se_cl"],
                             att=r["att"], att_se=r["att_se"], n=len(y)))
            if scheme == "random":
                keep[o] = dict(psi=r["psi"], e=r["e"], m1=r["m1"], m0=r["m0"])
                if o != "tenaga_kerja":
                    k = (r["e"] > .1) & (r["e"] < .9); d2 = dat[k].reset_index(drop=True)
                    r2 = fit_one(d2, y[k], b, scheme, seed)
                    rows.append(dict(split=s, scheme=scheme, outcome=o, check="trim10", ate=r2["ate"], se=r2["se_cl"], n=int(k.sum())))
                    m = (dat["pindah"] == 0).values; d3 = dat[m].reset_index(drop=True)
                    r3 = fit_one(d3, y[m], b, scheme, seed)
                    rows.append(dict(split=s, scheme=scheme, outcome=o, check="nonmover", ate=r3["ate"], se=r3["se_cl"], n=int(m.sum())))
                    r4 = fit_one(dat, y, b, scheme, seed, w=np.ones(len(dat)), X=XC)
                    rows.append(dict(split=s, scheme=scheme, outcome=o, check="unweighted", ate=r4["ate"], se=r4["se_cl"], n=len(y)))
    return rows, keep

res = Parallel(n_jobs=N_JOBS)(delayed(one_split)(s) for s in range(S_SPLIT))
X1raw = pd.DataFrame([r for rr, _ in res for r in rr]); X1raw.to_csv(os.path.join(OUT, "X1_splits_raw.csv"), index=False)
KEEP = [k for _, k in res]
agg = []
for (sc, o, ch), g in X1raw.groupby(["scheme", "outcome", "check"]):
    m, se = median_agg(g.ate, g.se)
    agg.append(dict(scheme=sc, outcome=o, check=ch, ate_median=m, se_adj=se, p=2 * (1 - stats.norm.cdf(abs(m / se))),
                    ate_min=g.ate.min(), ate_max=g.ate.max(), share_signif=float(np.mean(np.abs(g.ate / g.se) > 1.96)),
                    att_median=g.att.median() if g.att.notna().any() else np.nan, n=int(g.n.iloc[0]), S=len(g)))
X1 = pd.DataFrame(agg); X1.to_csv(os.path.join(OUT, "X1_repeated.csv"), index=False)
s0 = X1raw[(X1raw.split == 0) & (X1raw.scheme == "random") & (X1raw.outcome == "usaha_tutup") & (X1raw.check == "main")].ate.iloc[0]
log(f"Cek reproduksi split-0 (fold acak, usaha tutup): {s0:.4f}  (Tabel 3 run FULL: -0.0816; identik bila LightGBM)")
print(X1.round(4).to_string(index=False))

## X2. Overlap: trimming grid & overlap-weighted AIPW (rata-rata ψ berbobot e(1−e))

In [ ]:
rows = []
for o, b in OUTS.items():
    if o == "tenaga_kerja": continue
    y = dat[o].values.astype(float)
    for lo in [0.0, 0.02, 0.05, 0.10, 0.15]:
        est, ses, ns = [], [], []
        for s, kp in enumerate(KEEP):
            e = kp[o]["e"]; psi = kp[o]["psi"]; k = (e >= lo) & (e <= 1 - lo)
            m, se = cl_mean(psi[k], G[k], W[k]); est.append(m); ses.append(se); ns.append(int(k.sum()))
        m, se = median_agg(est, ses)
        rows.append(dict(outcome=o, estimand=f"ATE on {lo:.2f}<=e<={1-lo:.2f} (no refit)", est=m, se=se, n=int(np.median(ns))))
    est, ses = [], []
    for kp in KEEP:
        e = kp[o]["e"]; h = e * (1 - e); psi = kp[o]["psi"]
        m, se = cl_mean(psi, G, W * h); est.append(m); ses.append(se)
    m, se = median_agg(est, ses)
    rows.append(dict(outcome=o, estimand="Overlap-weighted AIPW, weights e(1-e)", est=m, se=se, n=len(y)))
    e_all = np.mean([kp[o]["e"] for kp in KEEP], axis=0)
    rows.append(dict(outcome=o, estimand="share e<0.1 or e>0.9", est=float(np.mean((e_all < .1) | (e_all > .9))), se=np.nan, n=len(y)))
X2 = pd.DataFrame(rows); X2["p"] = 2 * (1 - stats.norm.cdf(abs(X2.est / X2.se)))
X2.to_csv(os.path.join(OUT, "X2_overlap.csv"), index=False); print(X2.round(4).to_string(index=False))

## X3. Placebo pre-trend (negative-control outcome)
Sampel: RT yang punya usaha di 2000 **dan** 2007 (cocok via `hhid14`). Outcome = perubahan 2000→2007
(laba, aset, pekerja). Kredit 2014 tidak dapat menyebabkannya, dan—berbeda dengan plasebo "tutup 2000–07"—
outcome ini tidak menghilangkan kebutuhan kredit usaha. Kovariat: karakteristik 2000 saja + lokasi.

In [ ]:
EDU = {1: 0, 90: 0, 2: 6, 11: 6, 17: 6, 72: 6, 14: 6, 3: 9, 4: 9, 12: 9, 73: 9, 5: 12, 6: 12, 15: 12, 74: 12,
       60: 14, 13: 16, 61: 16, 62: 18, 63: 21, 10: 0}
if pl["pendidikan_kepala00"].max() > 25:
    pl["pendidikan_kepala00"] = pl["pendidikan_kepala00"].map(EDU)
    pl["pendidikan_kepala00"] = pl["pendidikan_kepala00"].fillna(pl["pendidikan_kepala00"].median())
c00 = [c for c in pl.columns if (c.endswith("00") or c.endswith("00_miss")) and c not in ("prov00", "kab00") and not c.startswith("hhid")]
ph = pl[pl["usaha_tutup_0007"] == 0].assign(_k=key(pl["hhid14"]))[["_k"] + c00].drop_duplicates("_k")
dh = dat.assign(_k=key(dat["hhid14"])).merge(ph, on="_k", how="inner").reset_index(drop=True)
log(f"Sampel pre-trend: {len(dh)} RT")
ok_w = (dh["pekerja07_miss"] == 0) & (dh["pekerja00_miss"] == 0) if "pekerja07_miss" in dh else (dh["pekerja00_miss"] == 0)
dh["d_ihs_laba_m"] = np.arcsinh(np.sinh(dh["ihs_laba07"]) / 1e6) - np.arcsinh(np.sinh(dh["ihs_laba00"]) / 1e6)
dh["d_ihs_aset_m"] = np.arcsinh(np.sinh(dh["ihs_aset07"]) / 1e6) - np.arcsinh(np.sinh(dh["ihs_aset00"]) / 1e6)
dh["d_pekerja"] = dh["pekerja07"] - dh["pekerja00"]
X00 = np.column_stack([dh[c00].values.astype(float), dh[["lon", "lat"]].values,
                       mundlak(dh[[c for c in c00 if not c.endswith("_miss")]].values.astype(float), dh["kab_kode"].values)])
rows = []
for o, sub in [("d_ihs_laba_m", np.ones(len(dh), bool)), ("d_ihs_aset_m", np.ones(len(dh), bool)), ("d_pekerja", ok_w.values)]:
    d2 = dh[sub].reset_index(drop=True); y = d2[o].values.astype(float); Xs = X00[sub]
    for wl in ["weighted", "unweighted"]:
        est, ses = [], []
        for s in range(S_SPLIT):
            seed = SEED + s * 7919 if s > 0 else SEED
            r = fit_one(d2, y, False, "random", seed, w=None if wl == "weighted" else np.ones(len(d2)), X=Xs)
            est.append(r["ate"]); ses.append(r["se_cl"])
        m, se = median_agg(est, ses)
        rows.append(dict(outcome=o, weights=wl, est=m, se=se, n=len(y), sd_outcome=float(np.std(y)),
                         std_effect=m / np.std(y)))
X3 = pd.DataFrame(rows); X3["p"] = 2 * (1 - stats.norm.cdf(abs(X3.est / X3.se)))
X3.to_csv(os.path.join(OUT, "X3_pretrend.csv"), index=False); print(X3.round(4).to_string(index=False))

## X4. Subgrup (dari ψ spesifikasi utama, dirata-rata lintas split)
Menjelaskan mengapa efek ≈ 0 pada sampel history-augmented: apakah efek terkonsentrasi pada usaha muda?

In [ ]:
in2000 = dat.assign(_k=key(dat["hhid14"]))["_k"].isin(set(ph["_k"])).values
age_ok = (dat.get("umur_usaha07_miss", pd.Series(0, index=dat.index)) == 0).values
groups = {
    "Enterprise existed in 2000 (history sample)": in2000,
    "Not in 2000 history sample": ~in2000,
    "Enterprise age >= 7 years (2007)": age_ok & (dat["umur_usaha07"] >= 7).values,
    "Enterprise age < 7 years (2007)": age_ok & (dat["umur_usaha07"] < 7).values,
    "Borrowed in 2007": (dat["pinjam07"] == 1).values,
    "Did not borrow in 2007": (dat["pinjam07"] == 0).values,
    "Urban 2007": (dat["perkotaan07"] == 1).values,
    "Rural 2007": (dat["perkotaan07"] == 0).values,
}
pairs = [(0, 1), (2, 3), (4, 5), (6, 7)]
rows = []
names = list(groups)
for o in ["usaha_tutup", "ihs_laba"]:
    psi = np.mean([kp[o]["psi"] for kp in KEEP], axis=0)
    for nm, msk in groups.items():
        m, se = cl_mean(psi[msk], G[msk], W[msk])
        rows.append(dict(outcome=o, group=nm, ate=m, se=se, n=int(msk.sum()), base_closure=float(np.average(dat.usaha_tutup[msk & (D == 0)], weights=W[msk & (D == 0)]))))
    for a, b in pairs:
        ma, mb = groups[names[a]], groups[names[b]]; sel = ma | mb
        Xm = np.column_stack([np.ones(sel.sum()), ma[sel].astype(float)])
        bb, se, _ = ols_cluster(psi[sel], Xm, G[sel], W[sel])
        rows.append(dict(outcome=o, group=f"DIFF: {names[a]} − {names[b]}", ate=bb[1], se=se[1], n=int(sel.sum())))
X4 = pd.DataFrame(rows); X4["p"] = 2 * (1 - stats.norm.cdf(abs(X4.ate / X4.se)))
X4.to_csv(os.path.join(OUT, "X4_subgroups.csv"), index=False); print(X4.round(4).to_string(index=False))

## X5. Sensitivitas Cinelli & Hazlett (2020): benchmark terhadap kovariat teramati
OLS/LPM + FE provinsi (WLS, t klasik seperti sensemakr; RV di notebook utama memakai t klaster sehingga sedikit lebih kecil). Konfounder hipotetis k× sekuat kovariat benchmark (pinjaman 2007, laba 2007, jarak lembaga).

In [ ]:
def wls_t(y, X, w):
    sw = np.sqrt(w); Xw = X * sw[:, None]; yw = y * sw
    XtXi = np.linalg.pinv(Xw.T @ Xw); b = XtXi @ Xw.T @ yw; u = yw - Xw @ b
    df_ = len(y) - X.shape[1]; s2 = (u @ u) / df_
    return b, np.sqrt(np.diag(XtXi) * s2), df_

prov_d = pd.get_dummies(dat["prov_kode"], drop_first=True).values.astype(float)
Xcov = dat[COVS].values.astype(float)
rows = []
for o in ["usaha_tutup", "ihs_laba", "tenaga_kerja"]:
    y = dat[o].values.astype(float)
    Xf = np.column_stack([np.ones(len(y)), D, Xcov, prov_d])
    b, se, dof = wls_t(y, Xf, W); est, sed = b[1], se[1]; t = est / sed
    r2yd = t ** 2 / (t ** 2 + dof); fq = abs(t) / math.sqrt(dof)
    rv = 0.5 * (math.sqrt(fq ** 4 + 4 * fq ** 2) - fq ** 2)
    rv05 = None
    fq2 = max(abs(t) - stats.t.ppf(0.975, dof), 0) / math.sqrt(dof)
    rv05 = 0.5 * (math.sqrt(fq2 ** 4 + 4 * fq2 ** 2) - fq2 ** 2)
    rows.append(dict(outcome=o, benchmark="(none)", k=np.nan, estimate=est, se=sed, r2_YD_given_X=r2yd, r2_DZ_given_X=np.nan, RV_q1=rv, RV_q1_alpha05=rv05,
                     adj_estimate=np.nan, adj_lo=np.nan, adj_hi=np.nan))
    for bm in ["pinjam07", "ihs_laba07", "jarak_lembaga_min07"]:
        j = COVS.index(bm)
        # partial R2 of benchmark with D | other X, and with Y | D, other X
        Xd = np.column_stack([np.ones(len(y)), Xcov, prov_d]); bd, sd_, dfd = wls_t(D.astype(float), Xd, W)
        td = bd[1 + j] / sd_[1 + j]; r2dxj = td ** 2 / (td ** 2 + dfd)
        ty = b[2 + j] / se[2 + j]; r2yxj = ty ** 2 / (ty ** 2 + dof)
        for k in [1, 2, 3]:
            r2dz = k * r2dxj / (1 - r2dxj)
            if r2dz >= 1: continue
            r2zxj = k * r2dxj ** 2 / ((1 - k * r2dxj) * (1 - r2dxj))
            r2yz = ((math.sqrt(k) + math.sqrt(r2zxj)) / math.sqrt(1 - r2zxj)) ** 2 * (r2yxj / (1 - r2yxj))
            r2yz = min(r2yz, 1)
            bias = sed * math.sqrt(r2yz * r2dz / (1 - r2dz)) * math.sqrt(dof)
            se_adj = sed * math.sqrt((1 - r2yz) / (1 - r2dz)) * math.sqrt(dof / (dof - 1))
            adj = est - np.sign(est) * bias
            rows.append(dict(outcome=o, benchmark=bm, k=k, estimate=est, se=sed, r2_YD_given_X=np.nan, r2_DZ_given_X=r2dz,
                             RV_q1=np.nan, RV_q1_alpha05=np.nan, adj_estimate=adj,
                             adj_lo=adj - 1.96 * se_adj, adj_hi=adj + 1.96 * se_adj))
X5 = pd.DataFrame(rows); X5.to_csv(os.path.join(OUT, "X5_sensitivity.csv"), index=False); print(X5.round(4).to_string(index=False))

## X6. BLP & GATES berulang (median lintas split; p-value = 2×median, Chernozhukov et al. 2025)

In [ ]:
CATE = ["ihs_laba07", "umur_usaha07", "kepala_perempuan", "pendidikan_kepala", "perkotaan07", "pinjam07",
        "jarak_lembaga_min07", "log_pce07"]
Xcate = dat[CATE].values.astype(float)
def het_split(s, o):
    kp = KEEP[s][o]; seed = SEED + s * 7919 if s > 0 else SEED
    f = make_folds("random", G, rxy(dat), N_FOLDS, seed=seed + 1)
    tau = dr_learner(kp["psi"], Xcate, f)
    blp = blp_test(kp["psi"], tau, G, W)
    gt = gates(kp["psi"], tau, G, 5, W)
    return dict(split=s, outcome=o, beta=blp["beta_het"], se=blp["se_het"], p=blp["p_het"],
                **{f"gate{int(r.group)}": r.gate for r in gt.itertuples()})
res6 = Parallel(n_jobs=N_JOBS)(delayed(het_split)(s, o) for s in range(S_SPLIT) for o in ["usaha_tutup", "ihs_laba"])
X6raw = pd.DataFrame(res6); X6raw.to_csv(os.path.join(OUT, "X6_blp_splits.csv"), index=False)
X6 = X6raw.groupby("outcome").agg(beta_median=("beta", "median"), se_median=("se", "median"),
                                   p_median_x2=("p", lambda x: min(1, 2 * np.median(x))),
                                   share_beta_negative=("beta", lambda x: float(np.mean(x < 0))),
                                   **{f"gate{g}_median": (f"gate{g}", "median") for g in range(1, 6)}).reset_index()
X6.to_csv(os.path.join(OUT, "X6_blp_repeated.csv"), index=False); print(X6.round(4).to_string(index=False))

## Ringkasan & ZIP

In [ ]:
lines = [f"# Eksperimen penutup (MODE={MODE}, S={S_SPLIT}, lightgbm={HAS_LGB})", ""]
for nm, df_ in [("X1 repeated cross-fitting", X1), ("X2 overlap", X2), ("X3 pre-trend placebo", X3),
                ("X4 subgroups", X4), ("X5 sensitivity", X5), ("X6 BLP repeated", X6)]:
    lines += [f"## {nm}", "", "```", df_.round(4).to_string(index=False), "```", ""]
open(os.path.join(OUT, "X_ringkasan.md"), "w").write("\n".join(lines))
import shutil
zp = shutil.make_archive(os.path.join("/tmp", f"X_results_{MODE}"), "zip", OUT)
shutil.move(zp, os.path.join(OUT, os.path.basename(zp)))
log("Selesai ->", os.path.join(OUT, os.path.basename(zp)))